<a href="https://colab.research.google.com/github/Mahid-Imran/flyrank-ml-internship-mahid-assignment2/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahid-Imran/flyrank-ml-internship-mahid-assignment2/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My selected lane is **Lane 2: Refresh / Content Opportunity Scoring**.

I frame this primarily as a **ranking task**. The business question is not simply whether a page belongs to a positive or negative class. The real question is which pages should be reviewed first when a content team has limited time.

The system would assign each eligible page a review-priority score. Pages would then be sorted from highest to lowest score to create a ranked review queue.

A classification model may be used underneath the ranking to estimate the probability that a page is showing meaningful decline or opportunity. However, the final business output is a ranking because an editor needs an ordered list rather than only a yes-or-no label.

The ranked output would support actions such as refreshing, expanding, protecting, merging, pruning, or monitoring a page.

In [6]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/Mahid-Imran/fly-rank-ml-assignment2.git"
REPO_DIR = "/content/fly-rank-ml-assignment2"

# Colab does not automatically download all repository files,
# so clone the repository when needed.
if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

# Create the initial slice used by the refresh-scoring lane.
lane_df = (
    df.loc[
        (df["impressions_90d"] > 0) &
        (df["content_age_days"] >= 90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)

print("Full dataset shape:", df.shape)
print("Lane dataset shape:", lane_df.shape)
print("Unique content items:", lane_df["content_id"].nunique())
print("Unique clients:", lane_df["client_id"].nunique())

Full dataset shape: (30000, 44)
Lane dataset shape: (30000, 44)
Unique content items: 30000
Unique clients: 32


## 2. Target or proxy

My provisional target is an **observed decline proxy**.

For the starter dataset, I will create a binary column called `decline_proxy`:

- `1` means the page's observed `trend_direction` is `down`;
- `0` means its observed trend direction is not `down`.

This is a derived proxy from measured performance in the starter snapshot. It is not a direct measurement of whether refreshing the page will improve its future performance.

Therefore, the target answers:

> Which pages are currently showing an observed declining trend?

It does not answer:

> Which pages are guaranteed to recover after an editor refreshes them?

A stronger final target would use an earlier feature window and a later outcome window from the daily warehouse data. For example, page information from one period could be used to predict whether impressions, clicks, or sessions decline in a later period.

Because `trend_direction` and `trend_pct` reveal the decline outcome, they must not be included as model features when this proxy is the target.

In [7]:
# Create a transparent provisional target.
lane_df["decline_proxy"] = (
    lane_df["trend_direction"].eq("down").astype("int8")
)

target_summary = (
    lane_df["decline_proxy"]
    .value_counts()
    .rename_axis("decline_proxy")
    .reset_index(name="number_of_pages")
)

target_summary["percentage"] = (
    target_summary["number_of_pages"] / len(lane_df) * 100
).round(2)

display(target_summary)

print(
    f"Positive target rate: "
    f"{lane_df['decline_proxy'].mean() * 100:.2f}%"
)

target_preview_columns = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "trend_direction",
    "decline_proxy"
]

display(lane_df[target_preview_columns].head(10))

,decline_proxy,number_of_pages,percentage
0,1,16262,54.21
1,0,13738,45.79


Positive target rate: 54.21%


,content_id,impressions_90d,clicks_90d,trend_direction,decline_proxy
0,content_304f48230142,3803,29,down,1
1,content_a1fb4e703a9e,15320,7,down,1
2,content_9aa793d4d895,12581,11,down,1
3,content_331d6c4de07b,11751,58,stable,0
4,content_d99b7a2d90ca,19140,24,down,1
5,content_d4084a4bc775,3970,1,down,1
6,content_9a34b442b552,20,0,down,1
7,content_a63219c6e95a,1724,1,stable,0
8,content_5e6c160719bc,32574,29,down,1
9,content_c27558df2b0c,1240,2,down,1


## 3. Success metric

My primary success metric is **Precision@50**.

Precision@50 measures how many of the top 50 pages in the ranked queue match the observed decline proxy.

For example:

- Precision@50 of `0.50` means that 25 of the top 50 recommended pages have a positive decline proxy;
- Precision@50 of `0.70` means that 35 of the top 50 pages have a positive decline proxy.

I selected this metric because the intended user has limited review capacity. The editor is more concerned with the quality of the pages at the top of the queue than with correctly classifying every page in the complete dataset.

For this provisional framing, I would consider the method useful if it achieves a Precision@50 of at least `0.50` and performs clearly better than a transparent fixed-rule baseline on held-out clients.

This metric evaluates whether the ranking finds observed candidates. It does not prove that editing the recommended pages will cause future improvement.

In [8]:
def precision_at_k(y_true, scores, k=50):
    """
    Calculate the proportion of positive targets among
    the k highest-scoring rows.
    """
    if len(y_true) != len(scores):
        raise ValueError("y_true and scores must have equal length.")

    if k <= 0:
        raise ValueError("k must be greater than zero.")

    k = min(k, len(y_true))

    evaluation = pd.DataFrame({
        "target": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = evaluation.nlargest(k, "score")
    return float(top_k["target"].mean())


# A simple temporary rule score only to demonstrate the metric.
# This is not the final model.
visibility_rank = lane_df["impressions_90d"].rank(pct=True)
staleness_rank = (
    lane_df["days_since_last_update"]
    .fillna(0)
    .rank(pct=True)
)

lane_df["demo_rule_score"] = (
    0.60 * visibility_rank +
    0.40 * staleness_rank
)

demo_precision_50 = precision_at_k(
    y_true=lane_df["decline_proxy"],
    scores=lane_df["demo_rule_score"],
    k=50
)

base_rate = lane_df["decline_proxy"].mean()

print(f"Overall positive target rate: {base_rate:.3f}")
print(f"Demonstration rule Precision@50: {demo_precision_50:.3f}")
print(
    f"Top-50 positive pages: "
    f"{round(demo_precision_50 * 50)} out of 50"
)

Overall positive target rate: 0.542
Demonstration rule Precision@50: 0.560
Top-50 positive pages: 28 out of 50


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized content item or page**.

Each dataframe row represents one unique `content_id`. The row contains observed information about that page, including visibility, clicks, CTR, average position, sessions, engagement, content age, freshness, and trend.

The system would produce one priority score for each row. The pages would then be sorted by that score to create the review queue.

The `content_id` and `client_id` fields are pseudonymized identifiers. They can be used for grouping, checking duplicates, and creating client-level validation splits, but they should not be used as predictive model features.

In [9]:
unit_columns = [
    "content_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "trend_direction",
    "decline_proxy"
]

unit_df = lane_df[unit_columns].copy()

# Confirm that one row represents one unique content item.
assert unit_df["content_id"].is_unique, (
    "content_id is not unique. Check the deduplication step."
)

print("One row = one unique content item/page")
print("Number of rows:", len(unit_df))
print("Unique content IDs:", unit_df["content_id"].nunique())
print("Duplicate content IDs:", unit_df["content_id"].duplicated().sum())

display(unit_df.head(10))

One row = one unique content item/page
Number of rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0


,content_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate,content_age_days,days_since_last_update,trend_direction,decline_proxy
0,content_304f48230142,keyword article,3803,29,0.76,10.6,17,5.88,187,20,down,1
1,content_a1fb4e703a9e,keyword article,15320,7,0.05,20.3,9,0.00,445,25,down,1
2,content_9aa793d4d895,keyword article,12581,11,0.09,36.5,11,0.00,141,20,down,1
3,content_331d6c4de07b,keyword article,11751,58,0.49,6.2,78,1.28,463,22,stable,0
4,content_d99b7a2d90ca,keyword article,19140,24,0.13,44.0,145,0.00,263,14,down,1
5,content_d4084a4bc775,keyword article,3970,1,0.03,8.5,5,0.00,147,20,down,1
6,content_9a34b442b552,keyword article,20,0,0.00,7.0,1,0.00,90,20,down,1
7,content_a63219c6e95a,keyword article,1724,1,0.06,21.2,28,3.57,445,22,stable,0
8,content_5e6c160719bc,keyword article,32574,29,0.09,46.0,68,5.88,90,20,down,1
9,content_c27558df2b0c,keyword article,1240,2,0.16,4.9,3,0.00,257,104,down,1


## 5. Why ML beats a fixed rule here

A fixed rule is an important baseline, but it may not be sufficient for the final ranking because page priority depends on several signals at the same time.

For example, an old page is not automatically a high-priority page. It may have very little visibility. Similarly, a page with low CTR may have weak position or too few impressions for its CTR to be reliable. A declining page may also have too little demand to justify immediate editorial work.

The relationship between impressions, position, CTR, age, freshness, content depth, sessions, and engagement may also be nonlinear. Different combinations may carry different levels of review priority.

A machine learning method may help by:

- combining several signals at the same time;
- learning interactions between them;
- producing a continuous priority score;
- ranking borderline cases more consistently;
- adapting better than one set of manually selected thresholds.

However, ML does not automatically beat a fixed rule. A transparent rule-based baseline should be created first. The ML method only earns its place if it produces better held-out ranking performance, especially better Precision@50, while remaining understandable through reason codes or feature explanations.

In [10]:
# Create several understandable rule flags.
lane_df["stale_visible_rule"] = (
    (lane_df["days_since_last_update"] >= 180) &
    (lane_df["impressions_90d"] >= 500)
)

lane_df["low_ctr_visible_rule"] = (
    (lane_df["impressions_90d"] >= 500) &
    (lane_df["avg_position"].between(1, 20)) &
    (lane_df["ctr"] < 0.5)
)

lane_df["thin_visible_rule"] = (
    (lane_df["word_count"] > 0) &
    (lane_df["word_count"] < 1200) &
    (lane_df["impressions_90d"] >= 250)
)

rule_columns = [
    "stale_visible_rule",
    "low_ctr_visible_rule",
    "thin_visible_rule"
]

rule_counts = (
    lane_df[rule_columns]
    .sum()
    .rename("number_of_flagged_pages")
    .to_frame()
)

display(rule_counts)

# Show how often the separate rules overlap.
lane_df["number_of_rules_triggered"] = (
    lane_df[rule_columns].sum(axis=1)
)

rule_overlap = (
    lane_df["number_of_rules_triggered"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_rules_triggered")
    .reset_index(name="number_of_pages")
)

display(rule_overlap)

print(
    "These rules create different and overlapping groups. "
    "This illustrates why one simple condition may not be "
    "enough to create the final ranking."
)

,number_of_flagged_pages
stale_visible_rule,17
low_ctr_visible_rule,9745
thin_visible_rule,82


,number_of_rules_triggered,number_of_pages
0,0,20184
1,1,9788
2,2,28


These rules create different and overlapping groups. This illustrates why one simple condition may not be enough to create the final ranking.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.